In [1]:

!pip -q install gradio pandas


In [2]:

import os
import sqlite3
from datetime import date, datetime

import gradio as gr
import pandas as pd

DB_PATH = "finance_tracker.db"


def init_db():
    with sqlite3.connect(DB_PATH) as conn:
        conn.execute("""
            CREATE TABLE IF NOT EXISTS transactions (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                txn_date TEXT NOT NULL,
                txn_type TEXT NOT NULL,
                category TEXT NOT NULL,
                description TEXT,
                amount REAL NOT NULL CHECK(amount > 0)
            )
        """)
        conn.execute("""
            CREATE TABLE IF NOT EXISTS budgets (
                month TEXT PRIMARY KEY,
                amount REAL NOT NULL CHECK(amount >= 0)
            )
        """)


def get_transactions():
    with sqlite3.connect(DB_PATH) as conn:
        return pd.read_sql_query(
            "SELECT id, txn_date AS Date, txn_type AS Type, "
            "category AS Category, description AS Description, "
            "amount AS Amount FROM transactions "
            "ORDER BY txn_date DESC, id DESC",
            conn
        )


def get_budget(month):
    with sqlite3.connect(DB_PATH) as conn:
        row = conn.execute(
            "SELECT amount FROM budgets WHERE month = ?", (month,)
        ).fetchone()
    return float(row[0]) if row else 0.0


def dashboard():
    tx = get_transactions()
    current_month = date.today().strftime("%Y-%m")

    if tx.empty:
        income = expense = 0.0
    else:
        tx["Amount"] = pd.to_numeric(tx["Amount"])
        tx["Date"] = tx["Date"].astype(str)
        current = tx[tx["Date"].str.startswith(current_month)]
        income = current.loc[current["Type"] == "Income", "Amount"].sum()
        expense = current.loc[current["Type"] == "Expense", "Amount"].sum()

    budget = get_budget(current_month)
    remaining = budget - expense if budget > 0 else None

    summary = f"""
# 💰 Personal Finance Tracker

### This Month
| Metric | Amount |
|---|---:|
| Income | ₹{income:,.2f} |
| Expenses | ₹{expense:,.2f} |
| Net Cash Flow | ₹{income-expense:,.2f} |
| Monthly Budget | {"₹{:,.2f}".format(budget) if budget else "Not set"} |
| Budget Remaining | {"₹{:,.2f}".format(remaining) if remaining is not None else "Set a budget"} |

*Net cash flow is this month's recorded income minus recorded expenses.
It is not your verified bank balance.*
"""
    if budget > 0 and expense > budget:
        summary += "\n\n⚠️ **You have exceeded your monthly budget.**"

    if tx.empty:
        categories = pd.DataFrame(columns=["Category", "Total Expenses"])
    else:
        expenses = tx[tx["Type"] == "Expense"]
        if expenses.empty:
            categories = pd.DataFrame(columns=["Category", "Total Expenses"])
        else:
            categories = (
                expenses.groupby("Category", as_index=False)["Amount"]
                .sum()
                .rename(columns={"Amount": "Total Expenses"})
                .sort_values("Total Expenses", ascending=False)
            )

    return summary, tx, categories


def add_transaction(txn_date, txn_type, category, description, amount):
    try:
        amount = float(amount)
        if amount <= 0:
            return "❌ Amount must be greater than zero.", *dashboard()

        if txn_date > date.today().isoformat():
            return "❌ Future transaction dates are not allowed.", *dashboard()

        if not category.strip():
            return "❌ Please select a category.", *dashboard()

        with sqlite3.connect(DB_PATH) as conn:
            conn.execute(
                "INSERT INTO transactions "
                "(txn_date, txn_type, category, description, amount) "
                "VALUES (?, ?, ?, ?, ?)",
                (txn_date, txn_type, category, description.strip(), amount)
            )

        summary, tx, categories = dashboard()
        return "✅ Transaction saved.", summary, tx, categories

    except (ValueError, TypeError):
        return "❌ Enter a valid amount.", *dashboard()


def save_budget(month, amount):
    try:
        amount = float(amount)
        if amount < 0:
            return "❌ Budget cannot be negative.", *dashboard()

        with sqlite3.connect(DB_PATH) as conn:
            conn.execute(
                "INSERT INTO budgets (month, amount) VALUES (?, ?) "
                "ON CONFLICT(month) DO UPDATE SET amount=excluded.amount",
                (month, amount)
            )

        summary, tx, categories = dashboard()
        return "✅ Monthly budget saved.", summary, tx, categories

    except (ValueError, TypeError):
        return "❌ Enter a valid budget.", *dashboard()


def delete_transaction(transaction_id):
    if transaction_id is None or str(transaction_id).strip() == "":
        return "❌ Enter the ID of the transaction to delete.", *dashboard()

    try:
        transaction_id = int(transaction_id)
        with sqlite3.connect(DB_PATH) as conn:
            cur = conn.execute(
                "DELETE FROM transactions WHERE id = ?", (transaction_id,)
            )
            if cur.rowcount == 0:
                return "❌ Transaction ID not found.", *dashboard()

        summary, tx, categories = dashboard()
        return "✅ Transaction deleted.", summary, tx, categories

    except ValueError:
        return "❌ Transaction ID must be a whole number.", *dashboard()


init_db()

with gr.Blocks(
    title="Personal Finance Tracker",
    theme=gr.themes.Soft(primary_hue="indigo", neutral_hue="slate"),
    css="""
    .gradio-container {max-width: 1100px !important; margin: auto}
    #header {
        padding: 24px; border-radius: 18px;
        background: linear-gradient(120deg,#172554,#4f46e5);
        color: white; margin-bottom: 16px;
    }
    #header h1, #header p {color: white !important}
    """
) as app:

    gr.HTML("""
    <div id="header">
      <h1>💰 Personal Finance Tracker</h1>
      <p>Track your income, control expenses, and manage your monthly budget.</p>
    </div>
    """)

    with gr.Tab("Dashboard"):
        summary_out = gr.Markdown()
        category_table = gr.Dataframe(
            label="Expense Totals by Category",
            interactive=False
        )
        refresh = gr.Button("Refresh Dashboard")
        refresh.click(
            fn=dashboard,
            inputs=[],
            outputs=[summary_out, transaction_table := gr.Dataframe(
                label="Transaction History",
                interactive=False,
                visible=False
            ), category_table]
        )

    with gr.Tab("Add Transaction"):
        txn_date = gr.Textbox(
            label="Transaction Date (YYYY-MM-DD)",
            value=date.today().isoformat()
        )
        txn_type = gr.Radio(
            ["Income", "Expense"], value="Expense", label="Transaction Type"
        )
        category = gr.Dropdown(
            [
                "Salary", "Business", "Other Income", "Food", "Transport",
                "Shopping", "Bills", "Education", "Health", "Rent",
                "Entertainment", "Savings", "Other Expense"
            ],
            value="Food",
            label="Category",
            allow_custom_value=True
        )
        description = gr.Textbox(
            label="Description", placeholder="e.g. groceries"
        )
        amount = gr.Number(label="Amount (₹)", minimum=0.01)
        add_btn = gr.Button("Save Transaction", variant="primary")
        add_status = gr.Markdown()

    with gr.Tab("Monthly Budget"):
        budget_month = gr.Textbox(
            label="Budget Month (YYYY-MM)",
            value=date.today().strftime("%Y-%m")
        )
        budget_amount = gr.Number(
            label="Monthly Expense Budget (₹)", minimum=0
        )
        budget_btn = gr.Button("Save Budget", variant="primary")
        budget_status = gr.Markdown()

    with gr.Tab("Transaction History"):
        history_table = gr.Dataframe(
            label="All Transactions", interactive=False
        )
        category_totals = gr.Dataframe(
            label="Expense Totals by Category", interactive=False
        )
        history_refresh = gr.Button("Refresh History")
        history_refresh.click(
            fn=dashboard,
            inputs=[],
            outputs=[summary_out, history_table, category_totals]
        )

        delete_id = gr.Number(label="Transaction ID to delete", precision=0)
        delete_btn = gr.Button("Delete Transaction")
        delete_status = gr.Markdown()

    add_btn.click(
        fn=add_transaction,
        inputs=[txn_date, txn_type, category, description, amount],
        outputs=[add_status, summary_out, history_table, category_totals]
    )

    budget_btn.click(
        fn=save_budget,
        inputs=[budget_month, budget_amount],
        outputs=[budget_status, summary_out, history_table, category_totals]
    )

    delete_btn.click(
        fn=delete_transaction,
        inputs=[delete_id],
        outputs=[delete_status, summary_out, history_table, category_totals]
    )

    app.load(
        fn=dashboard,
        inputs=[],
        outputs=[summary_out, history_table, category_totals]
    )

app.launch(share=True)


/tmp/ipykernel_3064/1717566248.py:170: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://fcc32144d53b4a71d9.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
